# Agentic Data Engineering Workshop
## Modern ETL + Data Quality + AI Agent Readiness

เป้าหมาย: สร้าง deterministic ETL pipeline จากข้อมูลยอดขายและสภาพอากาศ แล้วสร้าง Quality Report ที่สามารถส่งต่อให้ n8n / AI Agent วิเคราะห์ต่อได้

## 0) เตรียมข้อมูล
ใน Colab ให้ Upload ไฟล์ในโฟลเดอร์ `data/` ได้แก่ `sales_raw.csv`, `product_master.csv`, `weather_daily.json` หรือแก้ `DATA_DIR` ให้ชี้ไปยังโฟลเดอร์ของคุณ

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA_DIR = Path("data")  # แก้เป็น /content ถ้า upload ไฟล์ไว้ที่ root ของ Colab
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

## 1) Extract: อ่านข้อมูลจากหลายแหล่ง

In [ ]:
sales = pd.read_csv(DATA_DIR / "sales_raw.csv")
products = pd.read_csv(DATA_DIR / "product_master.csv")
weather = pd.read_json(DATA_DIR / "weather_daily.json")

print("sales", sales.shape)
print("products", products.shape)
print("weather", weather.shape)
display(sales)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Checkpoint 1
ตอบคำถาม: แถวใดมีปัญหาชัดเจน 3 อย่างแรกที่คุณเห็นคืออะไร?

In [ ]:
# Quick profile
sales.info()
sales.describe(include="all")

## 2) Transform: Normalize + Approved Mapping
AI สามารถช่วยเสนอการแก้คำผิดได้ แต่ใน pipeline จริงต้องใช้ mapping ที่อนุมัติแล้วเท่านั้น

In [ ]:
PRODUCT_CORRECTIONS = {"Coffe": "Coffee"}

df = sales.copy()
df["date_raw"] = df["date"]
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")
df["product_raw"] = df["product"]
df["product"] = df["product"].replace(PRODUCT_CORRECTIONS)
df["qty_num"] = pd.to_numeric(df["qty"], errors="coerce")
df["price_num"] = pd.to_numeric(df["price"], errors="coerce")
display(df)

## 3) Validate: ใช้กฎที่ตรวจสอบซ้ำได้

In [ ]:
known_products = set(products["product"])
flag_cols = []

df["is_duplicate"] = df.duplicated(subset=["order_id"], keep="first")
df["invalid_date"] = df["date"].isna()
df["missing_product"] = df["product"].isna() | (df["product"].astype("string").str.strip() == "")
df["unknown_product"] = ~df["product"].isin(known_products) & ~df["missing_product"]
df["missing_qty"] = df["qty_num"].isna()
df["negative_qty"] = df["qty_num"] < 0
df["non_numeric_price"] = df["price_num"].isna()
df["outlier_qty"] = df["qty_num"] > 100

flag_cols = ["is_duplicate","invalid_date","missing_product","unknown_product","missing_qty","negative_qty","non_numeric_price","outlier_qty"]
df["is_valid"] = ~df[flag_cols].any(axis=1)

display(df[["order_id","date_raw","date","product_raw","product","qty","price"] + flag_cols + ["is_valid"]])

### Checkpoint 2
เลือก 1 แถวที่ถูก reject และเขียนเหตุผลว่า reject เพราะกฎใด

## 4) Transform + Join
คำนวณ revenue และ join กับ Weather API ด้วย date

In [ ]:
valid = df[df["is_valid"]].copy()
valid["revenue"] = valid["qty_num"] * valid["price_num"]
weather["date"] = pd.to_datetime(weather["date"]).dt.date.astype("string")
clean = valid.merge(products, on="product", how="left").merge(weather, on="date", how="left")
clean = clean[["order_id","date","product","qty_num","price_num","channel","category","revenue","temperature_c","rain_mm","condition"]]
clean = clean.rename(columns={"qty_num":"qty","price_num":"price"})
display(clean)

## 5) Load: บันทึกผลลัพธ์

In [ ]:
clean.to_csv(OUT_DIR / "clean_sales_weather.csv", index=False)
df.to_csv(OUT_DIR / "sales_quality_flags.csv", index=False)
print("Saved:", OUT_DIR / "clean_sales_weather.csv")

## 6) Quality Report สำหรับ Agent

In [ ]:
report = {
    "pipeline": "daily_sales_weather",
    "total_rows": int(len(df)),
    "valid_rows": int(df["is_valid"].sum()),
    "rejected_rows": int((~df["is_valid"]).sum()),
}
for col in flag_cols:
    report[col] = int(df[col].sum())
report["severity"] = "HIGH" if report["rejected_rows"] / max(report["total_rows"], 1) > 0.2 else "LOW"
report["human_review_required"] = report["severity"] == "HIGH" or report["non_numeric_price"] > 0

with open(OUT_DIR / "quality_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))

## 7) Prompt สำหรับ AI Agent
ใช้ผลจาก Quality Report เป็นหลักฐาน ไม่ให้ AI เดาข้อมูลเอง

In [ ]:
prompt = f"""
You are a Data Quality Analyst for an ETL pipeline.

Analyze this ETL quality report and explain whether the pipeline output is safe to publish.
Do not change production data. Do not invent missing facts.

Quality report:
{json.dumps(report, ensure_ascii=False, indent=2)}

Return only valid JSON with keys:
severity, plain_language_summary, likely_causes, recommended_actions, safe_to_publish, human_review_required, message_to_team.
"""
print(prompt)

## 8) Optional: เรียก LLM ผ่าน API เมื่อมี API Key
เซลล์นี้เป็นตัวอย่างเท่านั้น หากไม่มี API Key ให้ copy prompt ไปใช้ในเครื่องมือ AI ที่ผู้สอนกำหนด

In [ ]:
# # Optional example, not required for the workshop.
# import os
# from google import genai
# from google.colab import userdata


# if userdata.get('GEMINI_API_KEY'):
#     client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
#     response = client.models.generate_content(model="gemini-3.6-flash", contents=prompt)
#     print(response.text)
# else:
#     print("No GEMINI_API_KEY found. Use the generated prompt manually.")

## Final reflection
1. ขั้นตอนใดควรใช้ deterministic rule?  
2. ขั้นตอนใดให้ AI ช่วยได้?  
3. ข้อมูลแบบใดต้องให้มนุษย์อนุมัติก่อนเผยแพร่?